# BreastCare: train the mammogram model on Kaggle

**Before running:** in the right-hand panel set *Accelerator* to **GPU T4 x2** and switch **Internet on**
(Kaggle asks for phone verification once). Then *Run All*. It takes roughly 30-60 minutes.

**What it does**
1. Downloads the starting model (ResNet50 mammography classifier, `umairinayat/medical-models`, MIT) and
   CBIS-DDSM full mammograms (CC and MLO views, CC BY 3.0) from Hugging Face, checking their SHA-256.
2. Splits the images **by patient** into train / validation / test, so no patient is in two splits.
3. Measures the unmodified starting model on the test patients (for comparison only).
4. Fine-tunes on the training patients; the best epoch and the decision threshold are chosen on validation.
5. Measures the trained model once on the untouched test patients: accuracy, precision, recall, specificity,
   F1, ROC-AUC, overall and for CC and MLO, with 95% intervals.
6. Exports `model.onnx` with those numbers inside, and checks that ONNX gives the same answers as PyTorch.

**Afterwards:** download `breastcare_model.zip` from the *Output* panel, unzip it into `ML-Service/weights/`
of the project (replacing `model.onnx`) and restart the backend. The website then runs this model on the
server's CPU and shows its test results to users.

In [ ]:
import os, pathlib
WORK = pathlib.Path('/kaggle/working/ML-Service') if os.path.isdir('/kaggle') else pathlib.Path('ML-Service-run')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
print(os.getcwd())

In [ ]:
%%writefile model.py
"""Model definition and preprocessing shared by train.py, evaluate.py and export_onnx.py.

The network is ResNet50. Training starts from the mammography classifier published at
https://huggingface.co/umairinayat/medical-models (disease_models/mammography/classification, MIT licence),
whose backbone has already seen mammograms, and is fine-tuned on full CBIS-DDSM mammograms (CC and MLO views).

Preprocessing must match the Spring Boot backend (OnnxPredictor.java) exactly, so both sides do the same thing:
grayscale -> bilinear resize to IMAGE_HEIGHT x IMAGE_WIDTH -> copy to 3 channels -> pixels in [0, 1].
The training images (CBIS-DDSM_1024) were squashed to a square from their portrait originals, so uploads are
squashed to a square the same way rather than padded. ImageNet normalisation and softmax live inside the
network (NormalizedClassifier), so the exported ONNX graph is the very module that was trained and evaluated.
"""
from pathlib import Path

import torch
from PIL import Image
from torchvision import models
from torchvision.transforms import functional as F

CLASSES = ["benign", "malignant"]
IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
ROOT = Path(__file__).parent
WEIGHTS_DIR = ROOT / "weights"
WEIGHTS_PATH = WEIGHTS_DIR / "model.pt"
METRICS_PATH = WEIGHTS_DIR / "metrics.json"
PRETRAINED_PATH = WEIGHTS_DIR / "pretrained" / "best_classifier.pt"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class NormalizedClassifier(torch.nn.Module):
    """Takes RGB pixels in [0, 1]; returns logits (training) or probabilities (probabilities=True)."""

    def __init__(self, backbone: torch.nn.Module):
        super().__init__()
        self.backbone = backbone
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, image: torch.Tensor, probabilities: bool = False) -> torch.Tensor:
        logits = self.backbone((image - self.mean) / self.std)
        return torch.softmax(logits, dim=1) if probabilities else logits


def build_model(init: str = "huggingface") -> NormalizedClassifier:
    """ResNet50 with a benign/malignant head.

    init="huggingface": backbone from the published mammography checkpoint; the new 2-class head starts from
    that model's benign and malignant rows. init="imagenet": plain ImageNet weights. init="none": random
    (only used to rebuild a model before loading a saved state dict).
    """
    backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if init == "imagenet" else None)
    backbone.fc = torch.nn.Linear(backbone.fc.in_features, len(CLASSES))
    if init == "huggingface":
        checkpoint = load_pretrained_checkpoint()
        state = checkpoint["model_state"]
        names = checkpoint["class_names"]
        rows = [names.index(name) for name in CLASSES]
        state = {**state, "fc.weight": state["fc.weight"][rows], "fc.bias": state["fc.bias"][rows]}
        backbone.load_state_dict(state)
    return NormalizedClassifier(backbone)


def load_pretrained_checkpoint() -> dict:
    """The Hugging Face checkpoint, loaded as plain tensors (weights_only: no pickled code is run)."""
    if not PRETRAINED_PATH.exists():
        raise SystemExit(f"{PRETRAINED_PATH} not found. Run: python download_pretrained.py")
    return torch.load(PRETRAINED_PATH, map_location="cpu", weights_only=True)


def load_trained(path: Path = WEIGHTS_PATH) -> tuple[NormalizedClassifier, dict]:
    """A model saved by train.py and its checkpoint (version, threshold, ...)."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model = build_model(init="none")
    model.load_state_dict(checkpoint["state_dict"])
    return model.eval(), checkpoint


def load_grayscale(path) -> Image.Image:
    return Image.open(path).convert("L")


def to_input(image: Image.Image) -> torch.Tensor:
    """The exact inference preprocessing: grayscale PIL image -> [3, H, W] tensor in [0, 1]."""
    resized = image.resize((IMAGE_WIDTH, IMAGE_HEIGHT), Image.BILINEAR)
    return F.to_tensor(resized).expand(3, -1, -1).contiguous()

In [ ]:
%%writefile download_pretrained.py
"""Download the published mammography classifier used as the starting point for training.

Source: https://huggingface.co/umairinayat/medical-models (MIT licence),
        disease_models/mammography/classification/best_classifier.pt (ResNet50, benign/malignant/normal).
Run:    python download_pretrained.py      -> weights/pretrained/best_classifier.pt (+ the author's metrics.json)

The file is checked against the SHA-256 Hugging Face publishes, and is later loaded with
torch.load(weights_only=True), so it can only contain tensors, never code.

Note: the author's reported 99.6% accuracy comes from a random split of heavily augmented lesion crops and is not
used anywhere in this project. Only evaluate.py's results on held-out full mammograms are shown to users.
"""
import hashlib
import urllib.request

from model import PRETRAINED_PATH

BASE = "https://huggingface.co/umairinayat/medical-models/resolve/main/disease_models/mammography/classification"
FILES = {
    "best_classifier.pt": "9e5d4f6be6d9ccc9f7eefa5ea0c05959b8de28423fd3c98fdd64750b0be3db2b",
    "metrics.json": None,  # the author's own numbers, kept for reference only
}


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def main():
    PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
    for name, expected in FILES.items():
        target = PRETRAINED_PATH.parent / name
        if not target.exists():
            print(f"downloading {name} ...")
            urllib.request.urlretrieve(f"{BASE}/{name}", target)
        if expected and sha256(target) != expected:
            target.unlink()
            raise SystemExit(f"{name}: checksum mismatch, file deleted. Try again.")
        print(f"ok {target}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prepare_cbis.py
"""Download and organise CBIS-DDSM full mammograms for training and evaluation.

Images:   dbaek111/CBIS-DDSM_1024 on Hugging Face - 3,086 full CBIS-DDSM mammograms (CC and MLO), 8-bit PNG,
          long side 1024 px, named by DICOM series UID.
Labels:   FLARE25-Agent-Xray/CBIS-DDSM dataset_detailed.csv - patient id, breast, view, pathology and the official
          CBIS-DDSM train/test split for every image, joined on the series UID.
Licence:  CBIS-DDSM is CC BY 3.0 (TCIA, https://doi.org/10.7937/K9/TCIA.2016.7O02S9CY); cite it. Research and
          education only.

Run:      python prepare_cbis.py              -> data/cbis/images/*.png and data/cbis/manifest.csv

Splits are by patient, so no patient's images are on both sides:
  test  = the official CBIS-DDSM test set, minus patients who also appear in the official training set
  val   = about 15% of the remaining training patients (used to pick the epoch and the decision threshold)
  train = the rest
Labels: MALIGNANT -> malignant; BENIGN and BENIGN_WITHOUT_CALLBACK -> benign.
"""
import argparse
import csv
import hashlib
import random
import urllib.request
import zipfile
from collections import defaultdict
from pathlib import Path

ROOT = Path(__file__).parent
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "cbis"
SOURCES = {
    "CBIS-DDSM_full_1024.zip": (
        "https://huggingface.co/datasets/dbaek111/CBIS-DDSM_1024/resolve/main/CBIS-DDSM_full_1024.zip",
        "123239b7f68c3a309b33784aff0a7f91ff9200edc5a2f76c4507ce96cc7c0e53",
    ),
    "dataset_detailed.csv": (
        "https://huggingface.co/datasets/FLARE25-Agent-Xray/CBIS-DDSM/resolve/main/dataset_detailed.csv",
        "bbe8a5f28dacaac065d68720e3f2cb0732e8a190438e0275fe978a4388a6cdd4",
    ),
}
VAL_FRACTION = 0.15
SEED = 20261028


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def fetch(name):
    url, expected = SOURCES[name]
    target = RAW / name
    if not target.exists():
        print(f"downloading {name} ...")
        RAW.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(url, target)
    if sha256(target) != expected:
        raise SystemExit(f"{target}: checksum mismatch. Delete it and run again.")
    return target


def dicom_uids(original_path):
    # "Mass-Training_P_00001_LEFT_CC/<study uid>/<series uid>/000000.dcm"; the PNGs are named by one of the two.
    return original_path.strip().split("/")[1:3]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--zip", type=Path, help="already downloaded CBIS-DDSM_full_1024.zip")
    parser.add_argument("--csv", type=Path, help="already downloaded dataset_detailed.csv")
    args = parser.parse_args()
    for path, name in ((args.zip, "CBIS-DDSM_full_1024.zip"), (args.csv, "dataset_detailed.csv")):
        if path:
            RAW.mkdir(parents=True, exist_ok=True)
            if not (RAW / name).exists():
                (RAW / name).write_bytes(path.read_bytes())

    meta = {}
    with open(fetch("dataset_detailed.csv"), newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            for uid in dicom_uids(row["original_path"]):
                meta[uid] = row

    images = OUT / "images"
    images.mkdir(parents=True, exist_ok=True)
    records = []
    with zipfile.ZipFile(fetch("CBIS-DDSM_full_1024.zip")) as archive:
        for info in archive.infolist():
            if info.is_dir() or not info.filename.endswith(".png"):
                continue
            split, folder, filename = info.filename.split("/")
            uid = filename.rsplit("_", 1)[0]
            row = meta.get(uid)
            if row is None:
                raise SystemExit(f"no metadata for {info.filename}")
            label = 1 if row["pathology"] == "MALIGNANT" else 0
            if label != (folder == "cancer") or split != row["split"]:
                raise SystemExit(f"label or split disagrees with the metadata for {info.filename}")
            target = images / f"{uid}.png"
            if not target.exists():
                target.write_bytes(archive.read(info))
            records.append({
                "file": f"images/{uid}.png",
                "patient_id": row["patient_id"],
                "breast": row["breast"],
                "view": row["view"],
                "abnormality": row["abnormality_type"],
                "pathology": row["pathology"],
                "label": label,
                "official_split": split,
            })

    train_patients = {r["patient_id"] for r in records if r["official_split"] == "train"}
    dropped = {r["patient_id"] for r in records if r["official_split"] == "test"} & train_patients

    # Validation: whole patients, drawn separately from patients with and without a malignant image.
    by_patient = defaultdict(list)
    for r in records:
        if r["official_split"] == "train":
            by_patient[r["patient_id"]].append(r)
    rng = random.Random(SEED)
    val_patients = set()
    for has_cancer in (True, False):
        group = sorted(p for p, rs in by_patient.items() if any(r["label"] for r in rs) == has_cancer)
        rng.shuffle(group)
        val_patients.update(group[: round(len(group) * VAL_FRACTION)])

    for r in records:
        if r["official_split"] == "test":
            r["split"] = "excluded" if r["patient_id"] in dropped else "test"
        else:
            r["split"] = "val" if r["patient_id"] in val_patients else "train"

    with open(OUT / "manifest.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(sorted(records, key=lambda r: (r["split"], r["patient_id"], r["file"])))

    print(f"{len(records)} images -> {OUT / 'manifest.csv'}")
    for split in ("train", "val", "test", "excluded"):
        rs = [r for r in records if r["split"] == split]
        patients = {r["patient_id"] for r in rs}
        views = {v: sum(r["view"] == v for r in rs) for v in ("CC", "MLO")}
        print(f"  {split:8s} {len(rs):5d} images  {len(patients):4d} patients  malignant {sum(r['label'] for r in rs):4d}  {views}")
    sides = [{r["patient_id"] for r in records if r["split"] == s} for s in ("train", "val", "test")]
    assert not (sides[0] & sides[1] or sides[0] & sides[2] or sides[1] & sides[2]), "a patient is in two splits"
    print("  no patient appears in more than one split")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile data_loading.py
"""CBIS-DDSM manifest (from prepare_cbis.py) as PyTorch datasets."""
import csv
from pathlib import Path

import torch
from torch.utils.data import Dataset
from torchvision import transforms

from model import IMAGE_HEIGHT, IMAGE_WIDTH, load_grayscale, to_input

DATA_DIR = Path(__file__).parent / "data" / "cbis"


def read_manifest(split, data_dir=DATA_DIR):
    manifest = data_dir / "manifest.csv"
    if not manifest.exists():
        raise SystemExit(f"{manifest} not found. Run: python prepare_cbis.py")
    with open(manifest, newline="", encoding="utf-8") as f:
        return [r for r in csv.DictReader(f) if r["split"] == split]


# Mild changes a real mammogram can show: either breast side, small positioning and exposure differences.
_augment = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=10, translate=(0.04, 0.04), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
])


class MammogramDataset(Dataset):
    def __init__(self, split, train=False, data_dir=DATA_DIR):
        self.rows = read_manifest(split, data_dir)
        self.data_dir = data_dir
        self.train = train

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        image = load_grayscale(self.data_dir / row["file"])
        if self.train:
            image = _augment(image.resize((IMAGE_WIDTH, IMAGE_HEIGHT)))
        return to_input(image), torch.tensor(int(row["label"]))

    @property
    def labels(self):
        return [int(r["label"]) for r in self.rows]

    @property
    def views(self):
        return [r["view"] for r in self.rows]

    @property
    def patients(self):
        return [r["patient_id"] for r in self.rows]

In [ ]:
%%writefile metrics.py
"""Classification metrics for benign (0) / malignant (1) scores, shared by train.py and evaluate.py."""
import numpy as np
from sklearn.metrics import roc_auc_score


def at_threshold(labels, scores, threshold):
    """Accuracy, precision, recall (sensitivity), specificity, F1, ROC-AUC and the confusion counts."""
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    predicted = (scores >= threshold).astype(int)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())

    def ratio(a, b):
        return a / b if b else 0.0

    precision = ratio(tp, tp + fp)
    recall = ratio(tp, tp + fn)
    return {
        "images": int(len(labels)),
        "malignant": int(labels.sum()),
        "benign": int(len(labels) - labels.sum()),
        "accuracy": ratio(tp + tn, len(labels)),
        "precision": precision,
        "recall": recall,  # sensitivity: share of malignant images the model flags
        "specificity": ratio(tn, tn + fp),  # share of benign images the model clears
        "f1": ratio(2 * precision * recall, precision + recall),
        "roc_auc": float(roc_auc_score(labels, scores)) if 0 < labels.sum() < len(labels) else None,
        "confusion_matrix": {"true_positive": tp, "false_positive": fp, "true_negative": tn, "false_negative": fn},
    }


def best_threshold(labels, scores):
    """The threshold with the highest balanced accuracy (Youden's J = sensitivity + specificity - 1).

    Chosen on the validation set only, never on the test set, so the test numbers stay honest.
    """
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    best, best_j = 0.5, -1.0
    for threshold in np.unique(np.round(scores, 4)):
        m = at_threshold(labels, scores, threshold)
        j = m["recall"] + m["specificity"] - 1
        if j > best_j:
            best, best_j = float(threshold), j
    return best


def patient_bootstrap(labels, scores, patients, threshold, rounds=1000, seed=0):
    """95% intervals for accuracy, recall, specificity and AUC, resampling whole patients.

    A patient's CC and MLO images are not independent, so resampling images would make the intervals too narrow.
    """
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    patients = np.asarray(patients)
    groups = [np.flatnonzero(patients == p) for p in np.unique(patients)]
    rng = np.random.default_rng(seed)
    keys = ("accuracy", "recall", "specificity", "roc_auc")
    samples = {k: [] for k in keys}
    for _ in range(rounds):
        index = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        m = at_threshold(labels[index], scores[index], threshold)
        for k in keys:
            if m[k] is not None:
                samples[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in samples.items() if v}


def report(labels, scores, views, patients, threshold, bootstrap=True):
    """Overall metrics plus the same metrics for CC and MLO images separately."""
    labels, scores, views = np.asarray(labels), np.asarray(scores), np.asarray(views)
    result = {"threshold": threshold, "overall": at_threshold(labels, scores, threshold), "by_view": {}}
    if bootstrap:
        result["overall"]["ci95"] = patient_bootstrap(labels, scores, patients, threshold)
    for view in ("CC", "MLO"):
        mask = views == view
        if mask.any():
            result["by_view"][view] = at_threshold(labels[mask], scores[mask], threshold)
            if bootstrap:
                result["by_view"][view]["ci95"] = patient_bootstrap(
                    labels[mask], scores[mask], np.asarray(patients)[mask], threshold)
    result["patients"] = int(len(set(patients)))
    return result

In [ ]:
%%writefile evaluate.py
"""Measure a model on the held-out CBIS-DDSM test set (patients never seen in training or validation).

Run:   python evaluate.py                -> weights/metrics.json for weights/model.pt (from train.py)
       python evaluate.py --hf-baseline  -> how the unmodified Hugging Face model does on the same images

metrics.json is what the website shows users (accuracy, precision, recall, specificity, F1, ROC-AUC, overall and
for CC and MLO separately, with 95% intervals). export_onnx.py embeds it in the model file, so the numbers
always belong to the model that is running.
"""
import argparse
import hashlib
import json
from datetime import datetime, timezone

import numpy as np
import torch
from torch.utils.data import DataLoader

import metrics
from data_loading import MammogramDataset
from model import METRICS_PATH, WEIGHTS_DIR, WEIGHTS_PATH, load_pretrained_checkpoint, load_trained

TEST_SET = {
    "name": "CBIS-DDSM full mammograms, official test split",
    "source": "TCIA CBIS-DDSM (CC BY 3.0) via Hugging Face dbaek111/CBIS-DDSM_1024 + FLARE25-Agent-Xray/CBIS-DDSM labels",
    "patient_disjoint": True,
    "note": "Patients who also appear in the training split were removed. Every image has a biopsy-proven "
            "benign or malignant finding; normal screening images are not part of CBIS-DDSM.",
}


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


@torch.no_grad()
def predict(model, dataset, device, batch_size=16, workers=4, amp=False):
    model.to(device).eval()
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=workers)
    scores = []
    for x, _ in loader:
        with torch.autocast(device.type, enabled=amp and device.type == "cuda"):
            scores += model(x.to(device), probabilities=True)[:, 1].float().cpu().tolist()
    scores = np.array(scores)
    if not np.isfinite(scores).all():
        raise SystemExit("The model produced NaN scores; it is broken and must not be used. Retrain it.")
    return scores


def hf_baseline(device):
    """The published 3-class model as it is: 224 px input, P(malignant) as the score."""
    from torchvision import models

    checkpoint = load_pretrained_checkpoint()
    backbone = models.resnet50()
    backbone.fc = torch.nn.Linear(backbone.fc.in_features, checkpoint["num_classes"])
    backbone.load_state_dict(checkpoint["model_state"])
    mean = torch.tensor(checkpoint["normalization_mean"]).view(1, 3, 1, 1)
    std = torch.tensor(checkpoint["normalization_std"]).view(1, 3, 1, 1)
    size = checkpoint["img_size"]
    names = checkpoint["class_names"]
    dataset = MammogramDataset("test")
    backbone.to(device).eval()
    probabilities = []
    with torch.no_grad():
        for x, _ in DataLoader(dataset, batch_size=32, num_workers=4):
            x = torch.nn.functional.interpolate(x, size=(size, size), mode="bilinear", antialias=True)
            probabilities.append(torch.softmax(backbone(((x - mean) / std).to(device)), 1).cpu())
    probabilities = torch.cat(probabilities).numpy()
    top = [names[i] for i in probabilities.argmax(1)]
    result = metrics.report(dataset.labels, probabilities[:, names.index("malignant")], dataset.views,
                            dataset.patients, 0.5)
    result["predicted_class_share"] = {n: top.count(n) / len(top) for n in names}
    return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--hf-baseline", action="store_true")
    args = parser.parse_args()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if args.hf_baseline:
        result = hf_baseline(device)
        out = WEIGHTS_DIR / "hf_baseline_metrics.json"
        out.write_text(json.dumps({"test_set": TEST_SET, **result}, indent=2))
        print(json.dumps(result, indent=2))
        print(f"wrote {out}")
        return

    model, checkpoint = load_trained()
    dataset = MammogramDataset("test")
    scores = predict(model, dataset, device)
    result = metrics.report(dataset.labels, scores, dataset.views, dataset.patients, checkpoint["threshold"])
    payload = {
        "model_version": checkpoint["version"],
        "model_sha256": file_sha256(WEIGHTS_PATH),
        "evaluated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "test_set": TEST_SET,
        "validation": checkpoint.get("validation"),
        **result,
    }
    METRICS_PATH.write_text(json.dumps(payload, indent=2))
    overall = result["overall"]
    print(json.dumps({k: overall[k] for k in ("accuracy", "precision", "recall", "specificity", "f1", "roc_auc")}, indent=2))
    for view, m in result["by_view"].items():
        print(f"{view}: acc={m['accuracy']:.3f} recall={m['recall']:.3f} spec={m['specificity']:.3f} auc={m['roc_auc']:.3f}")
    print(f"wrote {METRICS_PATH}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Fine-tune the mammography ResNet50 on full CBIS-DDSM mammograms (CC and MLO views).

Run once on a GPU - the easy way is the Kaggle notebook (kaggle_train.ipynb, see README), which runs all of this:
    python download_pretrained.py     # starting weights from Hugging Face
    python prepare_cbis.py            # data/cbis: images + patient-disjoint train/val/test manifest
    python train.py                   # -> weights/model.pt (best epoch by validation ROC-AUC)
    python evaluate.py                # -> weights/metrics.json, measured on the untouched test patients
    python export_onnx.py             # -> weights/model.onnx with the metrics inside, for the backend

Training is a one-time developer step. The website only runs the exported model.onnx on the server's CPU, so
visitors need no GPU.

The test split is never read here. The epoch is chosen by validation ROC-AUC, and the decision threshold by
validation balanced accuracy, so the test numbers in metrics.json are a fair estimate.
"""
import argparse
import json
import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader

import metrics
from data_loading import MammogramDataset
from evaluate import predict
from model import IMAGE_HEIGHT, IMAGE_WIDTH, WEIGHTS_DIR, WEIGHTS_PATH, build_model


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--accumulate", type=int, default=1, help="gradient steps per update (effective batch)")
    parser.add_argument("--no-amp", action="store_true",
                        help="train in fp32; needed on GTX 16xx cards, whose fp16 produces NaN")
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--patience", type=int, default=6, help="stop after this many epochs without a better val AUC")
    parser.add_argument("--init", choices=["huggingface", "imagenet"], default="huggingface")
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    seed_everything(args.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu":
        raise SystemExit("No CUDA GPU found. Train on Kaggle (kaggle_train.ipynb) instead of this computer.")
    torch.backends.cudnn.benchmark = True
    amp = not args.no_amp
    train_ds = MammogramDataset("train", train=True)
    val_ds = MammogramDataset("val")
    print(f"train {len(train_ds)} images, val {len(val_ds)} images, device {device}")

    loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=args.workers,
                        persistent_workers=args.workers > 0, drop_last=True)
    model = build_model(init=args.init).to(device)

    labels = np.array(train_ds.labels)
    # Weight the classes so the smaller one counts as much as the larger one.
    weights = torch.tensor([len(labels) / (2 * (labels == c).sum()) for c in (0, 1)], dtype=torch.float32)
    criterion = torch.nn.CrossEntropyLoss(weight=weights.to(device), label_smoothing=0.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    updates_per_epoch = len(loader) // args.accumulate
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=args.lr, epochs=args.epochs,
                                                    steps_per_epoch=updates_per_epoch, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=amp)

    WEIGHTS_DIR.mkdir(exist_ok=True)
    history, best_auc, stale = [], -1.0, 0
    for epoch in range(1, args.epochs + 1):
        started = time.time()
        model.train()
        total, seen = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, (x, y) in enumerate(loader, 1):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.autocast(device.type, enabled=amp):
                loss = criterion(model(x), y)
            if not torch.isfinite(loss):
                raise SystemExit("Loss became NaN/inf. On a GTX 16xx card use --no-amp; otherwise lower --lr.")
            scaler.scale(loss / args.accumulate).backward()
            if step % args.accumulate == 0 and step // args.accumulate <= updates_per_epoch:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            total += loss.item() * len(y)
            seen += len(y)

        scores = predict(model, val_ds, device, workers=args.workers, amp=amp)
        threshold = metrics.best_threshold(val_ds.labels, scores)
        val = metrics.at_threshold(val_ds.labels, scores, threshold)
        history.append({"epoch": epoch, "train_loss": total / seen, "val": val, "threshold": threshold})
        print(f"epoch {epoch:2d}  loss {total / seen:.4f}  val auc {val['roc_auc']:.4f}  acc {val['accuracy']:.3f}  "
              f"recall {val['recall']:.3f}  spec {val['specificity']:.3f}  thr {threshold:.3f}  "
              f"({time.time() - started:.0f}s)", flush=True)

        if val["roc_auc"] > best_auc:
            best_auc, stale = val["roc_auc"], 0
            torch.save({
                "state_dict": model.state_dict(),
                "version": f"resnet50-cbis-{args.init}-e{epoch}-valauc{val['roc_auc']:.3f}",
                "threshold": threshold,
                "validation": {"epoch": epoch, **val},
                "init": args.init,
                "image_size": [IMAGE_HEIGHT, IMAGE_WIDTH],
                "args": vars(args),
            }, WEIGHTS_PATH)
            print(f"  saved {WEIGHTS_PATH}")
        else:
            stale += 1
            if stale >= args.patience:
                print(f"no improvement for {args.patience} epochs, stopping")
                break

    (WEIGHTS_DIR / "training_history.json").write_text(json.dumps(history, indent=2))
    print(f"best val ROC-AUC {best_auc:.4f}. Next: python evaluate.py, then python export_onnx.py")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile export_onnx.py
"""Export the trained model to ONNX so Spring Boot can run it with ONNX Runtime on the server's CPU.

Run: python export_onnx.py           (after train.py and evaluate.py)  -> weights/model.onnx
     python export_onnx.py --demo    untrained placeholder, so the site can start before a model exists

The graph takes RGB pixels in [0, 1], shape [1, 3, IMAGE_HEIGHT, IMAGE_WIDTH], and returns
[benign, malignant] probabilities; normalisation and softmax are inside it, so Java only resizes and scales.

Metadata the backend reads: version, trained, classes, threshold, image_height, image_width and metrics (the
JSON from evaluate.py). Export stops if metrics.json was not measured on this exact model.pt, so the website can
never show numbers that belong to a different model.
"""
import argparse
import inspect
import json
import sys

import onnx
import torch

from evaluate import file_sha256
from model import CLASSES, IMAGE_HEIGHT, IMAGE_WIDTH, METRICS_PATH, WEIGHTS_PATH, build_model, load_trained

ONNX_PATH = WEIGHTS_PATH.with_suffix(".onnx")


class ProbabilityOutput(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, image):
        return self.model(image, probabilities=True)


def main():
    # The PyTorch exporter prints emoji, which a Windows console cannot encode.
    for stream in (sys.stdout, sys.stderr):
        stream.reconfigure(errors="replace")
    parser = argparse.ArgumentParser()
    parser.add_argument("--demo", action="store_true", help="export an untrained placeholder model")
    args = parser.parse_args()

    if args.demo:
        model, version, threshold, metrics = build_model(init="imagenet").eval(), "untrained-demo", 0.5, None
    else:
        if not WEIGHTS_PATH.exists():
            raise SystemExit(f"{WEIGHTS_PATH} not found. Run train.py first (or --demo for a placeholder).")
        if not METRICS_PATH.exists():
            raise SystemExit(f"{METRICS_PATH} not found. Run evaluate.py first.")
        metrics = json.loads(METRICS_PATH.read_text())
        if metrics.get("model_sha256") != file_sha256(WEIGHTS_PATH):
            raise SystemExit("metrics.json was measured on a different model.pt. Run evaluate.py again.")
        model, checkpoint = load_trained()
        version, threshold = checkpoint["version"], checkpoint["threshold"]

    ONNX_PATH.parent.mkdir(exist_ok=True)
    # Keep the weights inside the one .onnx file; older PyTorch versions (as on Kaggle) always do and lack the flag.
    options = {"external_data": False} if "external_data" in inspect.signature(torch.onnx.export).parameters else {}
    torch.onnx.export(
        ProbabilityOutput(model).eval(),
        torch.rand(1, 3, IMAGE_HEIGHT, IMAGE_WIDTH),
        ONNX_PATH,
        input_names=["image"],
        output_names=["probabilities"],
        **options,
    )

    onnx_model = onnx.load(ONNX_PATH)
    props = {
        "version": version,
        "trained": str(not args.demo).lower(),
        "classes": ",".join(CLASSES),
        "threshold": f"{threshold:.6f}",
        "image_height": str(IMAGE_HEIGHT),
        "image_width": str(IMAGE_WIDTH),
    }
    if metrics:
        props["metrics"] = json.dumps(metrics, separators=(",", ":"))
    for key, value in props.items():
        onnx_model.metadata_props.add(key=key, value=value)
    onnx.save(onnx_model, ONNX_PATH)
    print(f"exported {ONNX_PATH} (version={version}, trained={not args.demo}, threshold={threshold:.3f})")


if __name__ == "__main__":
    main()

In [ ]:
!pip install -q "onnx>=1.16" "onnxscript>=0.1" "onnxruntime>=1.18"
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 in the notebook settings"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
!python download_pretrained.py && python prepare_cbis.py

In [ ]:
# For comparison only: the published model as it is, on the same test patients.
!python evaluate.py --hf-baseline | tail -n 25

In [ ]:
!python train.py 2>&1 | tee weights/train.log

In [ ]:
!python evaluate.py && python export_onnx.py

In [ ]:
# The exported model must give the same probabilities as PyTorch on real test images.
import numpy as np, onnxruntime as ort, torch
from data_loading import MammogramDataset
from model import load_trained
model, _ = load_trained()
session = ort.InferenceSession('weights/model.onnx', providers=['CPUExecutionProvider'])
dataset = MammogramDataset('test')
worst = 0.0
for i in range(0, len(dataset), max(1, len(dataset) // 20)):
    x, _ = dataset[i]
    expected = model(x[None], probabilities=True).detach().numpy()
    actual = session.run(None, {'image': x[None].numpy()})[0]
    worst = max(worst, float(np.abs(expected - actual).max()))
print('largest difference', worst)
assert worst < 1e-3, 'ONNX export does not match PyTorch'

In [ ]:
import json, zipfile
m = json.load(open('weights/metrics.json'))
o = m['overall']
print(f"test patients {m['patients']}, images {o['images']} ({o['malignant']} malignant)")
for k in ('accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc'):
    print(f"{k:12s} {o[k]:.3f}   95% CI {o.get('ci95', {}).get(k, '')}")
for view, v in m['by_view'].items():
    print(view, {k: round(v[k], 3) for k in ('accuracy', 'recall', 'specificity', 'roc_auc')})
with zipfile.ZipFile('/kaggle/working/breastcare_model.zip' if os.path.isdir('/kaggle') else 'breastcare_model.zip', 'w') as z:
    for name in ('model.onnx', 'metrics.json', 'hf_baseline_metrics.json', 'training_history.json', 'train.log'):
        if (WORK / 'weights' / name).exists():
            z.write(WORK / 'weights' / name, name)
print('Download breastcare_model.zip from the Output panel and unzip it into ML-Service/weights/')

# Kaggle keeps everything in /kaggle/working as output; drop the 1.3 GB of data and the duplicate weights so the
# download is just the zip.
import shutil
for leftover in (WORK / 'data', WORK / 'weights' / 'pretrained', WORK / 'weights' / 'model.pt', WORK / 'weights' / 'model.onnx'):
    if leftover.is_dir():
        shutil.rmtree(leftover)
    elif leftover.exists():
        leftover.unlink()